In [3]:
import torch
import numpy as np
import pandas as pd
import networkx as nx
from tqdm import tqdm
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score
from collections import defaultdict 
from poincare import PoincareManifold
from model import Distance_PE
from data import G_hpo

In [4]:
# Modifier à chaque fois :
checkpoint = torch.load('logs/2026_5_7/12/model_final.pt', map_location='cpu', weights_only=False)

objects = checkpoint['objects']
node2id = checkpoint['node2id']
losses = checkpoint['losses']
norm_history = checkpoint['norm_history']
edges_closed = checkpoint['edges']
data = checkpoint['data']
hp = checkpoint['hyperparams']

edges = np.array([(node2id[u], node2id[v]) for u, v in G_hpo.edges()],dtype=np.int64)


In [5]:
print(hp)

{'dim': 15, 'epochs': 1500, 'lr': 1.6, 'burnin': 100, 'n_neg': 100}


In [4]:
manifold = PoincareManifold()
model = Distance_PE(n=len(objects), dim=hp['dim'],
                       manifold=manifold, sparse=False, learn_curvature=False, init_curvature=1., weight_decay=0)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

Distance_PE(
  (embeddings): Embedding(19389, 15)
)

In [5]:
W = model.weight.detach().cpu().numpy()   # (N, dim)
norms = np.linalg.norm(W, axis=1)

print(f"Modèle chargé — {len(objects)} nœuds | dim={hp['dim']} | "
      f"{len(losses)} epochs")
print(f"Norme moy={norms.mean():.4f} | max={norms.max():.4f}")

i_min = np.argmin(norms).item()
print(f"Index de la plus petite norme : {i_min}")
print(f"Position du point : {W[i_min]}")


Modèle chargé — 19389 nœuds | dim=15 | 1500 epochs
Norme moy=0.9640 | max=0.9900
Index de la plus petite norme : 101
Position du point : [-0.02290159  0.00820978 -0.00655441  0.00533598 -0.02455972 -0.06093883
 -0.00785416  0.03578258 -0.01433282  0.04514984  0.00592283 -0.01020399
 -0.04606588  0.02117609  0.04252743]


In [6]:
degrees = np.array([len(data.pos_neighbors[i]) for i in range(len(objects))])
norms = model.weight.detach().norm(dim=-1).numpy()

# Corrélation degré/norme attendue : négative
rho, pval = spearmanr(degrees, norms)
print(f"Corrélation Spearman degré/norme : {rho:.3f} (p={pval:.2e})")

Corrélation Spearman degré/norme : 0.196 (p=8.50e-168)


In [7]:
pos_neighbors = defaultdict(set)
pos_parents = defaultdict(set)

for u, v in edges:
    pos_neighbors[int(u)].add(int(v))
    pos_parents[int(v)].add(int(u))
    
len(pos_parents[i_min])
len(pos_parents[0])

7

In [8]:
degrees = np.array([len(data.pos_neighbors[i]) for i in range(len(objects))])

# Top 10 plus proches du centre
center_ids = np.argsort(norms)[:10]
print("=== 10 nœuds les plus proches du CENTRE ===")
for i in center_ids:
    print(f"  {objects[i]:<30} norme={norms[i]:.4f}  degré={degrees[i]}")

# Top 10 plus proches du bord
border_ids = np.argsort(norms)[-10:]
print("\n=== 10 nœuds les plus proches du BORD ===")
for i in border_ids:
    print(f"{objects[i]:<30} norme={norms[i]:.4f}  degré={degrees[i]}")

=== 10 nœuds les plus proches du CENTRE ===
  HP:0000118                     norme=0.1144  degré=1
  HP:0000001                     norme=0.1300  degré=0
  HP:0012649                     norme=0.2559  degré=1
  HP:0001939                     norme=0.2987  degré=1
  HP:0000163                     norme=0.3659  degré=1
  HP:0000818                     norme=0.4590  degré=1
  HP:0011793                     norme=0.4630  degré=1
  HP:0012823                     norme=0.4656  degré=1
  HP:0012638                     norme=0.4680  degré=1
  HP:0011001                     norme=0.4913  degré=1

=== 10 nœuds les plus proches du BORD ===
HP:0009509                     norme=0.9900  degré=3
HP:0009632                     norme=0.9900  degré=4
HP:0009535                     norme=0.9900  degré=2
HP:0009517                     norme=0.9900  degré=3
HP:0009529                     norme=0.9900  degré=3
HP:0009533                     norme=0.9900  degré=3
HP:0009669                     norme=0.9900  

In [9]:
@torch.no_grad()
def evaluate(model, objects, edges, node2id):
    model.eval()
    W = model.weight.to(device)
    pos_neighbors = defaultdict(set)
    for u, v in edges:
        pos_neighbors[int(u)].add(int(v))

    ranksum, ap_scores = 0, 0
    nranks = 0
    iters = 0
    labels = np.empty(model.embeddings.weight.size(0))
 
    for u in tqdm(objects):
        labels.fill(0)
        u = int(node2id[u])
        neighbors = pos_neighbors.get(u, set())
        if not neighbors :
            continue
        u_exp = W[u].unsqueeze(0).expand(W.shape[0], -1)  # Coordonnées de u dans la boule de Poincaré
        dists = manifold.distance(u_exp, W, 1).numpy()  # Distance de Poincaré de u aux autres noeuds
        dists[u] = 1e12
        #order = np.argsort(dists)  # Tri par distance décroissante p/r à u
        sorted_ind = np.argsort(dists)

        #ranks = int(np.where(order == v)[0][0]) + 1  # Rang du noeud v p/r à u dans l'embedding
        #ranks.append(rank)
        ranks, = np.where(np.isin(sorted_ind, list(neighbors)))
        ranks += 1
        N = ranks.shape[0]

        ranksum += ranks.sum() - (N * (N - 1) / 2)
        nranks += ranks.shape[0]
        labels[list(neighbors)] = 1
        ap_scores += average_precision_score(labels, -dists)
        iters += 1

        #pos  = pos_neighbors[u]  # Voisins de u dans la représentation initiale
        #hits, psum = 0, 0.0 
        #for k, idx in enumerate(order[1:], 1):  # On parcourt les noeuds du plus proche au plus éloigné
            #if idx in pos:
                #hits  += 1
                #psum  += hits / k
        #aps.append(psum / max(len(pos), 1))
 
    return float(ranksum), nranks, ap_scores, iters

In [10]:
objects_hpo = list(G_hpo.nodes())
node2id_hpo = {n: i for i, n in enumerate(objects_hpo)}
edges_hpo = np.array([(node2id_hpo[v], node2id_hpo[u]) for u, v in G_hpo.edges()],dtype=np.int64)
new_edges = [(v, u) for u, v in edges]
results = evaluate(model.weight.detach().cpu(), objects, new_edges, node2id)

print("Erreur moyenne sur le rang : ", float(results[0]) / results[1])
print("Mean Average Precision :", float(results[2]) / results[3])

AttributeError: 'Tensor' object has no attribute 'eval'

In [11]:
@torch.no_grad()
def evaluate2(model, objects, edges, node2id, device):
    """
    Retourne MAP et mean rank.
    Corrige : distance avec model.c, calcul GPU, pas de fuite u dans le ranking.
    """
    model.eval()
    W = model.weight.to(device)  # (N, dim)

    pos_neighbors = defaultdict(set)
    for u, v in edges:
        pos_neighbors[int(u)].add(int(v))

    ap_scores = []
    ranks_all = []
    N = W.shape[0]
    labels = np.zeros(N)

    for obj in tqdm(objects):
        u = int(node2id[obj])
        neighbors = pos_neighbors.get(u, set())
        if not neighbors:
            continue

        # Distances GPU avec la bonne courbure
        u_emb = W[u].unsqueeze(0).expand(N, -1)   # (N, dim)
        dists = model.manifold.distance(u_emb, W, model.c)  # (N,)
        dists[u] = float('inf')                    # exclure u lui-même
        dists_np = dists.cpu().numpy()

        max_finite = dists_np[np.isfinite(dists_np)].max()
        dists_np[~np.isfinite(dists_np)] = max_finite + 1.0

        # Rang des voisins
        sorted_ind = np.argsort(dists_np)
        ranks = np.where(np.isin(sorted_ind, list(neighbors)))[0] + 1
        # Correction : soustraire les rangs des autres voisins placés avant
        n_neighbors = len(neighbors)
        corrected_ranks = ranks - np.arange(n_neighbors)
        ranks_all.extend(corrected_ranks.tolist())

        # AP
        labels.fill(0)
        labels[list(neighbors)] = 1
        ap_scores.append(average_precision_score(labels, -dists_np))

    map_score  = float(np.mean(ap_scores))
    mean_rank  = float(np.mean(ranks_all))

    model.train()
    return map_score, mean_rank

In [12]:
results = evaluate2(model, objects, new_edges, node2id, device=torch.device("cuda" if torch.cuda.is_available() else "cpu"))

results

100%|██████████| 19389/19389 [00:37<00:00, 518.59it/s] 


(0.7653385461399612, 411.49415044135657)

In [13]:
print("min", norms.min(), "moy", norms.mean(), "max", norms.max())

min 0.11438677 moy 0.96395105 max 0.99000007


## Calcul de l'*information content* (des poids pour les barycentres)

In [6]:
profils_omim = pd.read_csv("../data/profils_omim.csv.gz", index_col=0)
profils_omim = profils_omim.reset_index()

/tmp/ipykernel_5230/2817799710.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  profils_omim = profils_omim.reset_index()


In [7]:
import re
import ast

hp_ids = []
parents_list = []

with open("../data/HPOs.csv", "r") as f:
    next(f)
    for line in f:
        hp_id = line.split(';')[0]
        
        # Extraire uniquement la liste contenant des IDs HP:XXXXXXX
        match = re.search(r"\[([^\]]*'HP:\d{7}'[^\]]*)\]", line)
        if match:
            parents = re.findall(r"HP:\d{7}", match.group(0))
        else:
            parents = []
        
        hp_ids.append(hp_id)
        parents_list.append(parents)

df_hpo = pd.DataFrame({'hp_id': hp_ids, 'parents': parents_list})

# Vérification
print(repr(df_hpo['hp_id'].iloc[1]))
print(df_hpo['parents'].iloc[1])
print(df_hpo['parents'].iloc[7])

'HP:0000002'
['HP:0001507']
['HP:0000014']


In [12]:
G_hpo_work = nx.DiGraph()
for hp_id in hp_ids:
    G_hpo_work.add_node(hp_id)
for hp_id, parents in zip(hp_ids, parents_list):
    for parent_id in parents:
        if parent_id in G_hpo_work:
            G_hpo_work.add_edge(hp_id, parent_id)

# Vérification
print([repr(n) for n in list(G_hpo_work.nodes)[:3]])
print("Ancêtres HP:6001462:", list(nx.ancestors(G_hpo_work, 'HP:6001462')))

["'HP:0000001'", "'HP:0000002'", "'HP:0000003'"]
Ancêtres HP:6001462: []


In [18]:
deprecated={
    'HP:0006887':'HP:0001249',
    'HP:0002275':'HP:0002311',
    'HP:0002370':'HP:0002311',
    'HP:0002438':'HP:0001317',
    'HP:0004059':'HP:0006433',
    'HP:0005365':'HP:0010976',
    'HP:0005435':'HP:0011840',
    'HP:0005807':'HP:0009881',
    'HP:0007543':'HP:0000962',
    'HP:0007680':'HP:0007894',
    'HP:0007850':'HP:0030666',
    'HP:0007898':'HP:0012231',
    'HP:0009062':'HP:0008936',
    'HP:0010064':'HP:0010091',
    'HP:0012178':'HP:0012177',
    'HP:0030050':'HP:0002524',  # Suspect
    'HP:0031014':'HP:0031632',
    'HP:0100786':'HP:0001262',
    'HP:0200065':'HP:0000533'
}

def get_ancestors0(G, node):
    visited = set()
    queue = list(G.successors(node))
    while queue:
        current = queue.pop()
        if current not in visited:
            visited.add(current)
            queue.extend(G.successors(current))
    return visited

# Test
print(get_ancestors0(G_hpo_work, 'HP:0000253'))
# doit donner {HP:0005484, HP:0000252, ..., HP:0000001}

def compute_information_content(df_omim, G_hpo, deprecated):
    colnames = df_omim.columns[1:]
    weights = defaultdict(float)
    diseases = defaultdict(set)
    all_diseases = defaultdict(set)
    ancestors = {}

    def get_ancestors(term):
        if term not in ancestors:
            ancestors[term]=get_ancestors0(G_hpo, term)
        return ancestors[term]

    for id, row in df_omim.iterrows():
        for term in colnames:
            if row[term]==1:
                resolved = deprecated.get(term, term)
                weights[resolved]+=1
                diseases[resolved].add(id)
                all_diseases[resolved].add(id)
                for ancestor in get_ancestors(resolved):
                    weights[ancestor]+=1
                    all_diseases[ancestor].add(id)
    total = sum(weights.values())
    return {t: w / total for t, w in weights.items()}, diseases, all_diseases


{'HP:0009121', 'HP:0033127', 'HP:0000252', 'HP:0012443', 'HP:0000118', 'HP:0002977', 'HP:0100547', 'HP:0000707', 'HP:0000234', 'HP:0000001', 'HP:0011842', 'HP:0000240', 'HP:0002011', 'HP:0012639', 'HP:0000929', 'HP:0005484', 'HP:0000924', 'HP:0040195', 'HP:0002060', 'HP:0007364', 'HP:0000152'}


In [19]:
hpo_cols = [col for col in profils_omim.columns if col.startswith("HP:")]
manquants = [col for col in hpo_cols if col not in G_hpo]
print(f"{len(manquants)} termes absents du graphe : {manquants}")

19 termes absents du graphe : ['HP:0002275', 'HP:0002370', 'HP:0002438', 'HP:0004059', 'HP:0005365', 'HP:0005435', 'HP:0005807', 'HP:0006887', 'HP:0007543', 'HP:0007680', 'HP:0007850', 'HP:0007898', 'HP:0009062', 'HP:0010064', 'HP:0012178', 'HP:0030050', 'HP:0031014', 'HP:0100786', 'HP:0200065']


In [20]:
weights, diseases, all_diseases = compute_information_content(profils_omim, G_hpo_work, deprecated)

In [21]:
# Vérifications
sum(weights.values())

1.0

In [24]:
def inspect_weights(weights, disease, all_disease, G_hpo, top_n=20):
    roots = [n for n in G_hpo.nodes if G_hpo.out_degree(n) == 0]
    root = roots[0] if roots else None

    G_inv = G_hpo.reverse()
    if root:
        depths = nx.single_source_shortest_path_length(G_inv, root)
    else:
        depths = {}

    rows = []
    for term, weight in weights.items():
        n_ancestors = len(get_ancestors0(G_hpo, term))
        n_children  = G_hpo.in_degree(term)
        rows.append({
            'term'       : term,
            'weight'     : weight,
            'depth'      : depths.get(term, -1),
            'n_ancestors': n_ancestors,
            'n_children' : n_children,
            'n_diseases' : len(disease.get(term, set())) if term in disease else 0,
            'n_diseases tot':len(all_disease.get(term, set())) if term in all_disease else 0
        })

    df = (pd.DataFrame(rows)
            .sort_values('weight', ascending=False)
            .reset_index(drop=True))

    df.index += 1
    df['weight'] = df['weight'].map('{:.6f}'.format)

    print(f"Racine détectée : {root}")
    print(f"Termes : {len(weights)}\n")
    print(df.head(top_n).to_string())
    return df

df_inspect = inspect_weights(weights, diseases, all_diseases, G_hpo_work, top_n=20)

Racine détectée : HP:0000001
Termes : 5674

          term    weight  depth  n_ancestors  n_children  n_diseases  n_diseases tot
1   HP:0000001  0.139856      0            0           7           0            6139
2   HP:0000118  0.139856      1            1          23        6139            6139
3   HP:0000707  0.033079      2            2           3        3877            3936
4   HP:0033127  0.030582      2            2           4        3798            3882
5   HP:0000924  0.021063      3            3           4        3005            3106
6   HP:0012638  0.020213      3            3          28        3581            3627
7   HP:0011842  0.019454      4            4          23        2899            2986
8   HP:0000152  0.016285      2            2           2        2828            3072
9   HP:0000234  0.015033      3            3           6        2781            3029
10  HP:0040064  0.011963      2            2          14        2184            2204
11  HP:0012639  0.011

'*HP:0000001*' correspond à la racine (et n'est pas présent dans `profils_omim`) et '*HP:0000118*' correspond à la catégorie *Phenotypic abnormality*, il est donc cohérent qu'elles aient les comptes les plus élevés, égaux au nombre de maladies.

## Représentation des maladies dans l'embeddings